In [ ]:
#pip install bertopic
#python -m spacy download es_core_news_sm

In [ ]:
import pandas as pd
import numpy as np
import json
import re
from typing import Dict, Any, Tuple, Union, Set
from umap import UMAP
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
import spacy

# Load data

In [ ]:
def extract_embedding(
    prompt_name: str,
    embedding_name: str,
    x: Union[str, Dict[str, Any]]
) -> np.ndarray:
    """
    Extract a specific embedding vector from a JSON-like object.

    Parameters
    ----------
    prompt_name : str
        Name of the prompt key inside `embedding_summary`.
    embedding_name : str
        Name of the embedding to extract (e.g., 'gemini_768').
    x : str or dict
        JSON content as a string or already decoded dictionary.

    Returns
    -------
    numpy.ndarray
        The corresponding embedding vector.

    Raises
    ------
    KeyError
        If the expected keys are not found in the JSON structure.
    """
    # Convert string → dict if needed
    if isinstance(x, str):
        data = json.loads(x)
    else:
        data = x

    return np.array(data["embedding_summary"][prompt_name][embedding_name])


def generate_embedding_array(
    df: pd.DataFrame,
    prompt_name: str,
    embedding_name: str
) -> np.ndarray:
    """
    Generate a 2D NumPy array of embeddings extracted from a DataFrame.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing a column named 'data' with JSON strings or dicts.
    prompt_name : str
        Name of the prompt key inside the embeddings structure.
    embedding_name : str
        Name of the embedding vector to extract.

    Returns
    -------
    numpy.ndarray
        A stacked 2D array with one embedding per row.
    """
    embeddings = df["data"].apply(
        lambda x: extract_embedding(prompt_name, embedding_name, x)
    ).tolist()

    return np.vstack(embeddings)

In [ ]:
def extract_summary(
    prompt_name: str,
    x: Union[str, Dict[str, Any]]
) -> str:
    """
    Extract investigator summary text from a JSON-like object.

    Parameters
    ----------
    prompt_name : str
        Name of the key inside the 'prompts' dictionary (e.g., 'system1_prompt1').
    x : str or dict
        JSON content either as a raw string or decoded dictionary.

    Returns
    -------
    str
        The extracted investigator summary, or an empty string if not available.
    """
    # Convert string → dict if needed
    if isinstance(x, str):
        try:
            data = json.loads(x)
        except json.JSONDecodeError:
            return ""
    else:
        data = x

    # Extract summary
    try:
        return data["prompts"][prompt_name]["resumen_investigador"]
    except KeyError:
        return ""


def generate_summary_df(
    df: pd.DataFrame,
    prompt_name: str
) -> pd.DataFrame:
    """
    Generate a DataFrame containing extracted summaries.

    Parameters
    ----------
    df : pandas.DataFrame
        Original DataFrame containing a column named 'data'.
    prompt_name : str
        Key inside 'prompts' from which to extract the summary.

    Returns
    -------
    pandas.DataFrame
        A DataFrame with one column named after the prompt, containing summaries.
    """
    summaries = df["data"].apply(lambda x: extract_summary(prompt_name, x))
    return pd.DataFrame({prompt_name: summaries})


In [ ]:
# Load file
df_emb = pd.read_parquet('summaries_embeddingsv1.parquet')

# Extract embeddings
embeddingsv1 = generate_embedding_array(df_emb, 'system1_prompt1', 'gemini_768')
embeddingsv2 = generate_embedding_array(df_emb, 'system2_prompt2_temp07', 'gemini_dim768')
embeddingsv3 = generate_embedding_array(df_emb, 'system3_prompt3_temp07', 'gemini_dim768')
embeddingsv4 = generate_embedding_array(df_emb, 'system4_prompt2_temp07', 'gemini_dim768')

# Extract summaries
df_summary = pd.DataFrame()
df_summary['system1_prompt1'] = generate_summary_df(df_emb, 'system1_prompt1')
df_summary['system2_prompt2_temp07'] = generate_summary_df(df_emb, 'system2_prompt2_temp07')
df_summary['system3_prompt3_temp07'] = generate_summary_df(df_emb, 'system3_prompt3_temp07')
df_summary['system4_prompt2_temp07'] = generate_summary_df(df_emb, 'system4_prompt2_temp07')

# BERTopic without UMAP

In [ ]:
# Load spaCy model only once (big performance improvement)
nlp = spacy.load("es_core_news_sm")

# Default stopwords from spaCy
STOPWORDS: Set[str] = nlp.Defaults.stop_words

# Domain-specific stopwords
DOMAIN_STOPWORDS: Set[str] = {
    "investigación", "investigar", "proyecto", "estudio", "campo", "área",
    "problema", "desarrollo", "trabajo", "metodología", "análisis",
    "objetivo", "resultados", "córdoba", "cordoba", "unc", "universidad",
    "facultad", "departamento", "instituto", "contexto", "marco",
    "proceso", "particular", "general", "principal", "diverso",
    "distinto", "relacionado", "vinculado"
}

# Unified stopwords set
ALL_STOPWORDS: Set[str] = STOPWORDS.union(DOMAIN_STOPWORDS)

# Precompiled repeated-pattern expressions (faster)
REPEATED_PATTERNS = [
    re.compile(r"el investigador[a-z\s,]*?", re.IGNORECASE),
    re.compile(r"la trayectoria investigadora[a-z\s,]*?", re.IGNORECASE),
    re.compile(r"la trayectoria de investigaci[oó]n[a-z\s,]*?", re.IGNORECASE),
    re.compile(r"este investigador[a-z\s,]*?", re.IGNORECASE),
    re.compile(r"su investigaci[oó]n[a-z\s,]*?", re.IGNORECASE),
]

# Precompiled unwanted character pattern
CLEAN_CHARS_PATTERN = re.compile(r"[^a-záéíóúñü\s]")


def clean_and_lemmatize(text: str) -> str:
    """
    Clean, normalize, and lemmatize Spanish text using spaCy.

    Steps performed:
    - Convert to lowercase
    - Remove repeated template expressions
    - Remove unwanted characters
    - spaCy lemmatization
    - Remove stopwords (general + domain-specific)
    - Remove very short tokens (<= 2 characters)

    Parameters
    ----------
    text : str
        The input Spanish text to clean and lemmatize.

    Returns
    -------
    str
        A cleaned and lemmatized string. Returns an empty string if the input
        is missing or not a valid text.
    """
    if not isinstance(text, str):
        return ""

    # Lowercase normalization
    text = text.lower()

    # Remove repeated narrative patterns
    for pattern in REPEATED_PATTERNS:
        text = pattern.sub(" ", text)

    # Remove disallowed characters
    text = CLEAN_CHARS_PATTERN.sub(" ", text)

    # Process with spaCy
    doc = nlp(text)

    # Lemmatize and filter
    tokens = []
    for token in doc:
        lemma = token.lemma_.strip()

        # Skip stopwords
        if lemma in ALL_STOPWORDS:
            continue

        # Skip extremely short tokens
        if len(lemma) <= 2:
            continue

        tokens.append(lemma)

    return " ".join(tokens)


In [ ]:
def extract_topics_bertopic(
    df: pd.DataFrame,
    text_column: str,
    embeddings: np.ndarray,
    min_topic_size: int = 10,
    n_gram_range: Tuple[int, int] = (1, 2),
    verbose: bool = True
) -> Tuple[BERTopic, pd.DataFrame]:

    """
    Extract high-quality thematic nodes (topics) from a dataframe column
    using BERTopic.

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe containing researcher text profiles.
    text_column : str
        Name of the column that contains the text (summaries/abstracts).
    embeddings : np.ndarray
        Embeddings for the text documents.
    min_topic_size : int
        Minimum cluster size to be considered a valid topic.
    n_gram_range : tuple
        N-gram range for keyphrase extraction (1 = unigrams, 2 = bigrams).
    verbose : bool
        If True, prints progress information.

    Returns
    -------
    model : BERTopic
        Fitted BERTopic model.
    topics_df : pandas.DataFrame
        DataFrame with topics and their main keywords.
    """

    # Topic model configuration (BERTopic using an HDBSCAN clustering)
    if verbose:
        print("[INFO] Fitting BERTopic model...")

    df[text_column] = df[text_column].apply(clean_and_lemmatize)

    topic_model = BERTopic(
        language="spanish",
        n_gram_range=n_gram_range,
        min_topic_size=min_topic_size,
        top_n_words=10,
        calculate_probabilities=True,
        verbose=verbose
    )

    docs = df[text_column].tolist()

    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)


    # Prepare output DataFrame
    if verbose:
        print("[INFO] Extracting topics...")

    topics_df = topic_model.get_topic_info()

    return topic_model, topics_df

# BERTopic with UMAP

In [ ]:
def extract_topics_bertopic_umap(
    df: pd.DataFrame,
    text_column: str,
    embeddings: np.ndarray,
    min_topic_size: int = 10,
    n_gram_range: Tuple[int, int] = (1, 2),
    umap_neighbors: int = 15,
    umap_min_dist: float = 0.0,
    umap_metric: str = "cosine",
    verbose: bool = True
) -> Tuple[BERTopic, pd.DataFrame]:
    """
    Extract high-quality thematic topics using BERTopic with tuned UMAP
    dimensionality reduction.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing researcher summaries.
    text_column : str
        Column name containing the text (Spanish/English).
    embeddings : np.ndarray
        Embeddings for the text documents.
    min_topic_size : int
        Minimum topic cluster size.
    n_gram_range : Tuple[int, int]
        N-gram range for keyphrase extraction.
    umap_neighbors : int
        UMAP parameter: larger values → more global structure.
    umap_min_dist : float
        UMAP parameter: lower values → denser clusters.
    umap_metric : str
        Distance metric for UMAP (typically "cosine").
    verbose : bool
        Print intermediate steps.

    Returns
    -------
    Tuple[BERTopic, pd.DataFrame]
        Fitted BERTopic model, DataFrame with extracted topic information
    """


    # Configure UMAP
    # Tuning UMAP significantly improves the quality of topic separation.
    if verbose:
        print("[INFO] Configuring UMAP...")

    umap_model: UMAP = UMAP(
        n_neighbors=umap_neighbors,
        min_dist=umap_min_dist,
        metric=umap_metric,
        random_state=42,
        verbose=verbose
    )


    # Configure and run BERTopic
    if verbose:
        print("[INFO] Fitting BERTopic model...")
    
    df[text_column] = df[text_column].apply(clean_and_lemmatize)

    topic_model: BERTopic = BERTopic(
        umap_model=umap_model,
        language="spanish",
        min_topic_size=min_topic_size,
        n_gram_range=n_gram_range,
        calculate_probabilities=True,
        verbose=verbose
    )

    docs = df[text_column].tolist()

    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)


    # Return topics
    topics_df: pd.DataFrame = topic_model.get_topic_info()

    return topic_model, topics_df

# Best result

In [4]:
# Dictionary mapping topic identifiers with system4_prompt2_temp07
# Note:
#   - The value -1 is used to represent noise or non-informative clusters.
#   - All non-negative integers correspond to meaningful thematic topics.
topic_dict = {
    -1: "Política, ambiente y sociedad — enfoques transversales",
     0: "Cultura, producción y análisis crítico social",
     1: "Estudios sociales y políticos del espacio urbano",
     2: "Química y caracterización de materiales",
     3: "Biología celular y modelos experimentales",
     4: "Diseño urbano y arquitectura del entorno construido",
     5: "Economía, modelos políticos y métodos analíticos",
     6: "Ecología, biodiversidad y cambio ambiental",
     7: "Salud clínica y salud mental — evaluación y diagnóstico",
     8: "Sostenibilidad, estrés hídrico y biomasa"
}

# Build a structured DataFrame from the topic dictionary.
df_topics = (
    pd.DataFrame(
        [
            {"topic_id": k, "description": v}
            for k, v in topic_dict.items()
        ]
    )
    .sort_values("topic_id")
    .reset_index(drop=True)
)

# Display only topic rows (excluding noise)
df_topics.loc[1:8]

,topic_id,description
1,0,"Cultura, producción y análisis crítico social"
2,1,Estudios sociales y políticos del espacio urbano
3,2,Química y caracterización de materiales
4,3,Biología celular y modelos experimentales
5,4,Diseño urbano y arquitectura del entorno construido
6,5,"Economía, modelos políticos y métodos analíticos"
7,6,"Ecología, biodiversidad y cambio ambiental"
8,7,Salud clínica y salud mental — evaluación y diagnóstico
